In [29]:
import pandas as pd
import numpy as np
import ast
import re
import nltk
from nltk.corpus import stopwords
import gensim.downloader as api

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Descargar stopwords de NLTK
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\marco\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# 1. Cargar el dataset
df = pd.read_csv('/workspace/tmdb_5000_movies.csv')

# Función para extraer el primer género de la cadena JSON
def extract_main_genre(genre_str):
    try:
        genres = ast.literal_eval(genre_str)
        if len(genres) > 0:
            return genres[0]['name']
        return None
    except:
        return None

# Aplicar la función y filtrar columnas relevantes
df['main_genre'] = df['genres'].apply(extract_main_genre)
df = df[['overview', 'main_genre']]

# 2. Eliminar filas con valores nulos
df = df.dropna(subset=['overview', 'main_genre']).reset_index(drop=True)

print(f"Total de películas después de limpieza: {len(df)}")
print(df.head(3))

Total de películas después de limpieza: 4772
                                            overview main_genre
0  In the 22nd century, a paraplegic Marine is di...     Action
1  Captain Barbossa, long believed to be dead, ha...  Adventure
2  A cryptic message from Bond’s past sends him o...     Action


In [31]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Convertir a minúsculas
    text = text.lower()
    # Eliminar caracteres especiales y puntuación (mantener solo letras y espacios)
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenización simple y eliminación de stopwords
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    # Reconstruir el string (requerido para el Tokenizer de Keras más adelante)
    return ' '.join(tokens)

df['clean_overview'] = df['overview'].apply(preprocess_text)
print("Ejemplo de texto limpio:")
print(df['clean_overview'].iloc[0])

Ejemplo de texto limpio:
nd century paraplegic marine dispatched moon pandora unique mission becomes torn following orders protecting alien civilization


In [32]:
print("Cargando modelo GloVe...")
glove_model = api.load("glove-wiki-gigaword-100")
print("¡Modelo GloVe cargado exitosamente!")

Cargando modelo GloVe...
¡Modelo GloVe cargado exitosamente!


In [33]:
# Función para calcular el promedio de los embeddings de una sinopsis
def get_average_embedding(text, embedding_model, vector_size=100):
    words = text.split()
    # Extraer el vector de cada palabra si existe en GloVe
    vectors = [embedding_model[word] for word in words if word in embedding_model]
    
    # Si encontramos palabras en GloVe, calculamos el promedio
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    # Si la sinopsis está vacía o no tiene palabras en GloVe, devolvemos ceros
    else:
        return np.zeros(vector_size)

print("Calculando promedios de embeddings para cada sinopsis...")
X_avg = np.array([get_average_embedding(text, glove_model, 100) for text in df['clean_overview']])

# Dividimos los datos promediados (Para el Modelo Inicial)
X_train_avg, X_test_avg, y_train, y_test = train_test_split(
    X_avg, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Forma de X_train_avg: {X_train_avg.shape} -> (Muestras, Vector Promediado)")

Calculando promedios de embeddings para cada sinopsis...
Forma de X_train_avg: (3817, 100) -> (Muestras, Vector Promediado)


In [34]:
print("--- ENTRENANDO MODELO INICIAL (BASE) ---")

# Arquitectura simple basada en los embeddings promediados
modelo_base = Sequential([
    Dense(64, activation='relu', input_shape=(100,)),
    Dense(num_classes, activation='softmax')
])

modelo_base.compile(loss='sparse_categorical_crossentropy', 
                    optimizer='adam', 
                    metrics=['accuracy'])

# Entrenamos el modelo base
history_base = modelo_base.fit(
    X_train_avg, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_avg, y_test),
    verbose=0 # verbose=0 para no llenar el notebook de texto, solo veremos los resultados
)

# Predicciones del modelo base
y_pred_base = np.argmax(modelo_base.predict(X_test_avg), axis=1)

print("\nEntrenamiento del Modelo Base finalizado.")

--- ENTRENANDO MODELO INICIAL (BASE) ---


c:\Users\marco\.conda\envs\unir\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

Entrenamiento del Modelo Base finalizado.


In [35]:
# Para el modelo optimizado con LSTM, necesitamos la SECUENCIA (no el promedio)
max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(df['clean_overview'])

X_seq = tokenizer.texts_to_sequences(df['clean_overview'])
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')

X_train_seq, X_test_seq, _, _ = train_test_split(X_pad, y, test_size=0.2, random_state=42, stratify=y)

# Matriz de embeddings de GloVe para la capa Embedding
embedding_matrix = np.zeros((max_words, 100))
for word, i in tokenizer.word_index.items():
    if i < max_words:
        try:
            embedding_matrix[i] = glove_model[word]
        except KeyError:
            pass

In [36]:
print("--- ENTRENANDO MODELO OPTIMIZADO (LSTM + Dropout + Class Weights) ---")

# 1. Calcular pesos de clase para el desbalanceo
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

# 2. Arquitectura Optimizada
modelo_optimizado = Sequential([
    Embedding(input_dim=max_words, output_dim=100, weights=[embedding_matrix], input_length=max_len, trainable=False),
    LSTM(128, return_sequences=False),
    Dense(64, activation='relu'),
    Dropout(0.5), # Técnica de optimización: Dropout
    Dense(num_classes, activation='softmax')
])

optimizer = Adam(learning_rate=0.001)
modelo_optimizado.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

# 3. Entrenamiento con pesos ponderados
history_opt = modelo_optimizado.fit(
    X_train_seq, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_test_seq, y_test),
    class_weight=class_weight_dict, # Técnica de optimización: Ponderación
    verbose=0
)

# Predicciones del modelo optimizado
y_pred_opt = np.argmax(modelo_optimizado.predict(X_test_seq), axis=1)

print("\nEntrenamiento del Modelo Optimizado finalizado.")

--- ENTRENANDO MODELO OPTIMIZADO (LSTM + Dropout + Class Weights) ---


c:\Users\marco\.conda\envs\unir\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step

Entrenamiento del Modelo Optimizado finalizado.


In [37]:
from sklearn.metrics import f1_score, accuracy_score

labels_indices = np.arange(len(label_encoder.classes_))
target_names = label_encoder.classes_

print("======================================================")
print("             COMPARACIÓN DE RENDIMIENTO               ")
print("======================================================\n")

print("1. RESULTADOS MODELO INICIAL (Promedio Embeddings + Dense)")
print("-" * 50)
print(classification_report(y_test, y_pred_base, labels=labels_indices, target_names=target_names, zero_division=0))

print("\n2. RESULTADOS MODELO OPTIMIZADO (Secuencias + LSTM + Dropout + Class Weights)")
print("-" * 50)
print(classification_report(y_test, y_pred_opt, labels=labels_indices, target_names=target_names, zero_division=0))

# Comparación cuantitativa directa (Macro F1 y Accuracy)
f1_base = f1_score(y_test, y_pred_base, average='macro', zero_division=0)
f1_opt = f1_score(y_test, y_pred_opt, average='macro', zero_division=0)

acc_base = accuracy_score(y_test, y_pred_base)
acc_opt = accuracy_score(y_test, y_pred_opt)

print("======================================================")
print("                 ANÁLISIS DE MEJORA                   ")
print("======================================================")
print(f"-> Accuracy Modelo Inicial:    {acc_base:.4f}")
print(f"-> Accuracy Modelo Optimizado: {acc_opt:.4f}")
print(f"-> Macro F1 Modelo Inicial:    {f1_base:.4f}")
print(f"-> Macro F1 Modelo Optimizado: {f1_opt:.4f}")
print("======================================================")
print("CONCLUSIÓN PARA EL REPORTE:")
print("El modelo base tiende a ignorar las clases minoritarias debido al desbalanceo extremo.")
print("El modelo optimizado, gracias a 'class_weights', obliga a la red a prestar atención a las clases pequeñas,")
print("lo que generalmente incrementa el Recall de clases difíciles, aunque puede afectar la precisión general.")
print("La LSTM procesa el contexto secuencial (frente a la pérdida de información de promediar los embeddings).")

             COMPARACIÓN DE RENDIMIENTO               

1. RESULTADOS MODELO INICIAL (Promedio Embeddings + Dense)
--------------------------------------------------
                 precision    recall  f1-score   support

         Action       0.47      0.52      0.49       151
      Adventure       0.37      0.34      0.35        68
      Animation       0.57      0.16      0.25        25
         Comedy       0.45      0.65      0.54       209
          Crime       0.44      0.18      0.25        39
    Documentary       0.83      0.29      0.43        17
          Drama       0.44      0.59      0.50       241
         Family       0.00      0.00      0.00        11
        Fantasy       0.00      0.00      0.00        24
        Foreign       0.00      0.00      0.00         0
        History       0.00      0.00      0.00         5
         Horror       0.38      0.42      0.40        60
          Music       0.00      0.00      0.00         7
        Mystery       0.00      0.0